# Ukraine air-alert forecasts: compact Poisson revision

A completed Python project for civilian planning. The model forecasts Kyiv city alert starts on the next UTC calendar day, issued at preceding noon. Version 0.2 reduces features, uses positive Poisson expected counts, and separates validation selection from locked final evaluation. The original historical test was already scored; it is explicitly a reused holdout, excluded from this revision's tuning.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'reports' / 'metrics.json').exists():
    if (ROOT.parent / 'reports' / 'metrics.json').exists():
        ROOT = ROOT.parent
    elif (ROOT / 'ukraine_alerts' / 'reports' / 'metrics.json').exists():
        ROOT = ROOT / 'ukraine_alerts'
    else:
        raise FileNotFoundError('Run inside the project or notebooks directory')
metrics = json.loads((ROOT / 'reports' / 'metrics.json').read_text())
selection = json.loads((ROOT / 'reports' / 'selection.json').read_text())
diagnosis = json.loads((ROOT / 'reports' / 'diagnosis.json').read_text())
predictions = pd.read_csv(ROOT / 'reports' / 'predictions.csv')
print(metrics['task'])
print('Frozen data cutoff:', metrics['data_as_of'])

Kyiv City next UTC calendar-day alert-start count; issued at preceding day 12:00 UTC
Frozen data cutoff: 2026-10-10 03:19:39+00:00


## Diagnosis before the final test

A 28-day mean changes slowly. It was a legitimate validation winner, not a stuck estimator. The larger learner had a high feature-to-row ratio, while available counts changed in level. Repeated command-line runs also default to the same frozen origin: they cannot create new information. See the README for explicit origins and newer-vintage acquisition.

In [2]:
print('Old numeric/text budget:', diagnosis['old_numeric_features'], diagnosis['old_text_cap'])
print('Initially available training labels:', metrics['training_rows_at_first_validation_origin'])
print('Initial/validation count means:', diagnosis['available_initial_training_mean'], diagnosis['validation_mean'])
print('New maximum numeric/text budgets:', metrics['feature_budget'])

Old numeric/text budget: 54 300
Initially available training labels: 139
Initial/validation count means: 1.85 3.75
New maximum numeric/text budgets: {'history_numeric_cap': 10, 'news_numeric_addition': 3, 'text_cap': 50, 'text_per_training_row_cap': 'floor(n_train/3)', 'selected_total_min': 63, 'selected_total_max': 63}


## Validation-only model choice

There are 36 compact Poisson candidates: history/news, direct/recent-level-offset, three alphas, and three training windows. The vocabulary is fitted per origin, with at most 50 unigrams and at most one coordinate per three eligible rows. Ten history/calendar inputs and three optional news aggregates replace 54 numeric coordinates.

A recent-level model uses `lambda = exposure * exp(intercept + X beta)`, with exposure from past seven-day starts plus a fixed half-count. Fitting y/exposure with sample weight exposure is the corresponding count Poisson likelihood. Predictions are not clipped. Averages are benchmark comparisons; the requested deployment pool comprises Poisson models.

In [3]:
print('Validation:', selection['validation'])
print('Selection instant:', selection['selection_at'])
print('Test rows exposed to selection:', selection['test_rows_seen_during_selection'])
print('Chosen:', selection['selected_config'])
print('Chosen validation scores:', selection['trials'][selection['selected_model']]['scores'])
print('Benchmark validation scores:', selection['benchmarks'])
print('Saved selection pool:', selection['selection_pool'])

Validation: {'start': '2026-08-13 00:00:00+00:00', 'end': '2026-09-09 00:00:00+00:00', 'days': 28, 'labels': 'first usable vintage, published before selection'}
Selection instant: 2026-09-10 12:00:00+00:00
Test rows exposed to selection: 0
Chosen: {'name': 'history_news_poisson__recent7__alpha1__windowall', 'with_news': True, 'alpha': 1.0, 'train_window': None, 'level_offset': True}
Chosen validation scores: {'MAE': 2.3071974946067564, 'RMSE': 2.8715371139764927, 'Poisson_deviance': 2.744810238569496}
Benchmark validation scores: {'rolling28': {'MAE': 2.1862244897959187, 'RMSE': 2.6478682497853954, 'Poisson_deviance': 2.2763928373721716}, 'rolling7': {'MAE': 2.2908163265306123, 'RMSE': 2.7823108460980537, 'Poisson_deviance': 2.3742186711108197}, 'weekday8': {'MAE': 2.2544642857142856, 'RMSE': 2.884398457960639, 'Poisson_deviance': 2.940200208304836}}
Saved selection pool: compact Poisson models, as requested; averages are separate benchmarks


## Earlier training-only analysis

The descriptive chart uses only outcomes available at the first validation origin. Calendar and historical features are computed as of each earlier forecast, not from the newest revised snapshot.

![Initial training series and autocorrelation](../reports/training_analysis.png)

## Locked final evaluation

All 25 tests passed before scoring. The selected pipeline/configuration was locked and checked before the final stage. Daily refits can incorporate earlier labels only after publication, under unchanged chosen settings. Final revised truth is used for scoring, not as a training target.

In [4]:
print(pd.DataFrame(metrics['test_metrics']).T.round(4).to_string())
print('Selected forecast variation:', metrics['prediction_range'])
print('Selection unchanged during scoring:', metrics['selection_unchanged_during_test'])
print('Historical holdout reused:', metrics['historical_holdout_reused'])
assert (predictions.selected_prediction > 0).all()
print('All selected expected counts are positive.')

                         MAE    RMSE  Poisson_deviance
rolling28             1.2972  1.5132            0.5325
rolling7              1.3929  1.6897            0.6653
weekday8              1.4866  1.8536            0.9116
history_poisson       1.4720  1.7927            0.7859
history_news_poisson  1.5178  1.8104            0.8003
Selected forecast variation: {'min': 3.0210321711089065, 'max': 4.6630260621610375, 'std': 0.4638332110605198, 'distinct_rounded_2dp': 24}
Selection unchanged during scoring: True
Historical holdout reused: True
All selected expected counts are positive.


![Locked forecasts and benchmark](../reports/evaluation.png)

The selected Poisson produces 24 distinct two-decimal forecasts, ranging 3.02–4.66 starts/day. Its MAE is 1.518; the average benchmark is better at 1.297. More variation does not establish more predictive information. The sampled news has no demonstrated benefit, and this short reused historical holdout cannot establish independent generalization.

In [5]:
print('News minus history Poisson MAE:', metrics['news_minus_history_mae'])
print('Exploratory paired bootstrap interval:', metrics['news_minus_history_mae_block_bootstrap95'])
print((ROOT / 'reports' / 'tests_before_final.log').read_text().split('----------------------------------------------------------------------')[-1].strip())

News minus history Poisson MAE: 0.04578878707243117
Exploratory paired bootstrap interval: [-0.05878751956174268, 0.10625412494641408]
Ran 25 tests in 9.808s

OK


## Inspect the timing and fit audits

In [6]:
audit = json.loads((ROOT / 'reports' / 'feature_audit.json').read_text())
fields = ['alert_available_at', 'max_news_published_at', 'max_news_available_at', 'max_news_indexed_at', 'max_batch_available_at']
checks = 0
for row in audit:
    for field in fields:
        if row[field] is not None:
            assert pd.Timestamp(row[field]) < pd.Timestamp(row['origin'])
            checks += 1
fits = json.loads((ROOT / 'reports' / 'training_audit.json').read_text())
for fit in fits:
    assert pd.Timestamp(fit['max_train_label_available_at']) < pd.Timestamp(fit['origin'])
    assert pd.Timestamp(fit['max_train_target_end']) < pd.Timestamp(fit['origin'])
    assert fit['text_feature_count'] <= min(50, fit['training_rows'] // 3)
print('Source-before-origin checks:', checks)
print('Final-period fit checks passed:', len(fits))

Source-before-origin checks: 985
Final-period fit checks passed: 28


## Reproduce

Run `python -m ua_alerts.evaluate --stage select`, then `python -m unittest discover -s tests -v`, then `python -m ua_alerts.evaluate --stage test`. Read `reports/selection.json` before scoring and `reports/metrics.json` afterward. `python -m ua_alerts.forecast` reconstructs the latest eligible frozen-origin forecast; `--origin` requests another allowed noon instant. See README for refresh instructions. Official alerts determine protective action.